In [ ]:
# Full EDA + Model Selection + Feature Importance Pipeline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Optional: Install libraries for some algorithms
# !pip install xgboost lightgbm catboost

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# ========================
# 0. Load & Clean Data
# ========================
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

train = train.replace([np.inf, -np.inf], np.nan)
test = test.replace([np.inf, -np.inf], np.nan)

# ========================
# 1. Train-Test Split
# ========================
X = train.drop(columns=["SalePrice"])
y = train["SalePrice"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Shapes:")
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

# ========================
# 2. EDA
# ========================
plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
sns.histplot(train["SalePrice"], bins=50, kde=True)
plt.title("Distribution of SalePrice (Raw)")
plt.xlabel("SalePrice")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(train["SalePrice"]), bins=50, kde=True, color="orange")
plt.title("Distribution of SalePrice (Log-transformed)")
plt.xlabel("Log(SalePrice)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

numeric_feats = train.select_dtypes(include=[np.number])
corr = numeric_feats.corr()["SalePrice"].sort_values(ascending=False)
top10_features = corr.index[1:11]

plt.figure(figsize=(10, 8))
sns.heatmap(numeric_feats[top10_features].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap of Top 10 Features with SalePrice")
plt.show()

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
sns.boxplot(x="Neighborhood", y="SalePrice", data=train)
plt.xticks(rotation=90)
plt.title("SalePrice by Neighborhood (Raw)")
plt.ylabel("SalePrice")

plt.subplot(1, 2, 2)
sns.boxplot(x="Neighborhood", y=np.log1p(train["SalePrice"]), data=train)
plt.xticks(rotation=90)
plt.title("SalePrice by Neighborhood (Log-transformed)")
plt.ylabel("Log(SalePrice)")
plt.tight_layout()
plt.show()

# ========================
# 3. Preprocessing
# ========================
numeric_features = X.select_dtypes(include=[np.number]).columns
categorical_features = X.select_dtypes(exclude=[np.number]).columns

numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")),
                                      ("scaler", StandardScaler())])
categorical_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")),
                                          ("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# ========================
# 4. Model Selection and Training
# ========================
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    "LightGBM": LGBMRegressor(n_estimators=100, random_state=42),
    "CatBoost": CatBoostRegressor(iterations=100, verbose=0, random_state=42)
}

rmse_scores = {}

for name, model_instance in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor),
                               ("regressor", model_instance)])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))
    rmse_scores[name] = rmse
    print(f"{name} RMSE: {rmse:.2f}")

# ========================
# 5. Select Best Model
# ========================
best_model_name = min(rmse_scores, key=rmse_scores.get)
print("\nBest Model:", best_model_name)
print("Best RMSE:", rmse_scores[best_model_name])

# Train the best model on full training data
best_model_pipeline = Pipeline(steps=[("preprocessor", preprocessor),
                                      ("regressor", models[best_model_name])])
best_model_pipeline.fit(X_train, y_train)

# ========================
# 6. Feature Importance (Top 10)
# ========================
# Works only for tree-based models
if best_model_name in ["Decision Tree", "Random Forest", "Gradient Boosting", "XGBoost", "LightGBM", "CatBoost"]:
    # Get feature names after preprocessing
    onehot_columns = best_model_pipeline.named_steps['preprocessor'] \
                                        .named_transformers_['cat'] \
                                        .named_steps['onehot'].get_feature_names_out(categorical_features)
    feature_names = np.concatenate([numeric_features, onehot_columns])
    
    importances = best_model_pipeline.named_steps['regressor'].feature_importances_
    importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
    importance_df = importance_df.sort_values(by="Importance", ascending=False).head(10)
    
    plt.figure(figsize=(10,6))
    sns.barplot(x="Importance", y="Feature", data=importance_df)
    plt.title(f"Top 10 Feature Importances ({best_model_name})")
    plt.show()
else:
    print("Feature importance is not available for this model.")

# ========================
# 7. Evaluation Metrics for Best Model
# ========================
y_pred_best = best_model_pipeline.predict(X_valid)
mse_best = mean_squared_error(y_valid, y_pred_best)
rmse_best = np.sqrt(mse_best)
r2_best = r2_score(y_valid, y_pred_best)

print("\nBest Model Evaluation:")
print(f"R² Score: {r2_best:.4f}")
print(f"Mean Squared Error (MSE): {mse_best:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_best:.2f}")

# ========================
# 8. Generate Predictions for Test Set and Create Submission
# ========================

# Apply best model pipeline to test data
# Make sure the test set has the same preprocessing
test_preds = best_model_pipeline.predict(test)

# Convert predictions to a DataFrame in submission format
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": test_preds
})

# Optional: apply log1p if you want to submit log-transformed values
# But Kaggle expects raw SalePrice, so we submit as is
# RMSE will be computed on log-transformed internally by Kaggle

# Save to CSV
submission_file = "submission.csv"
submission.to_csv(submission_file, index=False)
print(f"Submission file saved as {submission_file}")

# Display first few rows
submission.head()


